# Protein fitness landscape analysis

We’ll analyse an empirical protein fitness landscape using real experimental measurements, from preparing the data and constructing a landscape to exploring its ruggedness, epistasis and navigability. The GB1 deep mutational scanning (DMS) dataset will serve as our example.

## 1. Importing necessary dependencies

Install GraphFLA with pip, then import the dependencies below.

In [ ]:
!pip install graphfla

In [1]:
import graphfla
from graphfla.landscape import ProteinLandscape
import pandas as pd

## 2. Loading the experimental dataset

For our example, we use the GB1 dataset from [Wu et al. (2016)](https://doi.org/10.7554/eLife.16965). The experiment varied four positions in protein G domain B1: **39, 40, 41 and 54**. Fitness measures performance in an IgG-Fc binding selection, reflecting both folding and binding. Higher scores indicate better performance, with the wild-type variant `VDGV` assigned a score of 1.

The full 56-residue GB1 reference sequence, excluding assay tags, is:

```text
MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE
```

Each row in the CSV represents one variant. The `sequences` column contains only the four mutated positions; the remaining residues are fixed. We label the amino-acid columns with `residue_` followed by the one-based protein position.

| Column | Meaning |
|---|---|
| `sequences` | Amino acids at positions 39, 40, 41 and 54, in that order. |
| `residue_39` | Amino acid at position 39; V in the reference. |
| `residue_40` | Amino acid at position 40; D in the reference. |
| `residue_41` | Amino acid at position 41; G in the reference. |
| `residue_54` | Amino acid at position 54; V in the reference. |
| `fitness` | Relative assay score; larger is better. |

Let’s load the CSV from GitHub and look at the first few rows. The rename maps older column labels to the protein’s residue numbers.

In [2]:
data_url = (
    "https://raw.githubusercontent.com/COLA-Laboratory/GraphFLA/"
    "main/data/BioSequence/Wu2016_GB1.csv"
)
df = pd.read_csv(data_url).rename(columns={
    "pos1": "residue_39",
    "pos2": "residue_40",
    "pos3": "residue_41",
    "pos4": "residue_54",
})
df.head()

,sequences,residue_39,residue_40,residue_41,residue_54,fitness
0,VDGV,V,D,G,V,1.000000
1,ADGV,A,D,G,V,0.061910
2,CDGV,C,D,G,V,0.242237
3,DDGV,D,D,G,V,0.006472
4,EDGV,E,D,G,V,0.032719


## 3. Preparing the inputs

To construct a landscape, GraphFLA only needs two inputs—much like the features `X` and targets `y` used to train a machine-learning model:

- `X`: the protein variants, represented as sequence strings, a DataFrame with one amino-acid column per position, or a two-dimensional NumPy array.
- `f`: the measured fitness scores, supplied as a Series, list or one-dimensional array. This plays the role of `y` in the analogy.

The rows of `X` and `f` must refer to the same variants in the same order. We can start by selecting the sequence and fitness columns:

In [3]:
X = df["sequences"]
f = df["fitness"]
X.head()

0    VDGV
1    ADGV
2    CDGV
3    DDGV
4    EDGV
Name: sequences, dtype: object

We can also use the four residue columns as a DataFrame. Let’s use this form for construction so the residue names carry through to the results.

In [4]:
X = df[["residue_39", "residue_40", "residue_41", "residue_54"]]
X.head()

,residue_39,residue_40,residue_41,residue_54
0,V,D,G,V
1,A,D,G,V
2,C,D,G,V
3,D,D,G,V
4,E,D,G,V


## 4. Constructing the landscape

With the inputs ready, we can build a `ProteinLandscape`. We set `maximize=True` because higher fitness is better in this experiment.

By default, neighbours differ at one amino-acid position, and improving edges point from lower to higher fitness. Equal-fitness neighbours are stored as neutral pairs; connected neutral variants form a *plateau*.

In [5]:
landscape = ProteinLandscape(maximize=True)
landscape.build_from_data(X, f)

Building Landscape from data...


Preparing sequence input for Sequence (AY)...


Detected DataFrame/ndarray format input.


Sequence input preparation complete.


 - Handling missing values and duplicates...


...

   - Found 184 local optima (186 member nodes: 2 plateau-LOs + 182 single-point LOs).


 - Determining global optimum...


   - Global optimum found at index 57022 with fitness 8.7620.


Landscape built successfully.



ProteinLandscape(maximize=True)

## 5. Inspecting the landscape

Let’s first look at what we’ve built. Printing the landscape shows its variable positions, variants, improving edges and peaks:

In [6]:
print(landscape)

ProteinLandscape(kind='protein'): 4 variables, 149361 configurations, 5074326 edges, 184 local optima


For a closer look at individual variants, we can call `get_data()`. The returned table includes fitness and graph attributes: `out_degree` counts directly improving moves, `in_degree` counts incoming moves from lower-fitness neighbours, and `is_lo` indicates peak membership.

In [7]:
landscape.get_data()

,residue_39,residue_40,residue_41,residue_54,fitness,plateau_id,plateau_size,in_degree,out_degree,is_lo
0,V,D,G,V,1.000000,-1,1,56,20,False
1,A,D,G,V,0.061910,-1,1,30,30,False
2,C,D,G,V,0.242237,-1,1,50,25,False
3,D,D,G,V,0.006472,-1,1,26,48,False
4,E,D,G,V,0.032719,-1,1,40,35,False
...,...,...,...,...,...,...,...,...,...,...
149356,Y,Y,Y,R,0.001350,-1,1,18,57,False
149357,Y,Y,Y,S,0.004421,1456,2,21,53,False
149358,Y,Y,Y,T,0.021200,-1,1,37,38,False
149359,Y,Y,Y,W,0.009136,-1,1,59,14,False


We can also summarise fitness with pandas:

In [8]:
landscape.get_data()["fitness"].describe()

count    149361.000000
mean          0.080510
std           0.395391
min           0.000000
25%           0.001370
50%           0.003384
75%           0.008986
max           8.761966
Name: fitness, dtype: float64

To inspect the best observed variant, we can use its node index:

In [9]:
landscape[landscape.go_index]

{'residue_39': 'F',
 'residue_40': 'W',
 'residue_41': 'A',
 'residue_54': 'A',
 'fitness': np.float64(8.761965656),
 'plateau_id': -1,
 'plateau_size': 1,
 'in_degree': 75,
 'out_degree': 0,
 'is_lo': True}

## 6. Analysing the landscape

Next, we’ll count peaks, measure ruggedness, examine epistasis and assess the trend toward the best variant. All four analyses use the full experimental dataset.

In [10]:
from graphfla import analysis

### 6.1 Number of peaks

We can start by counting peaks with `landscape.n_lo`. A peak is a local fitness optimum; each connected neutral peak plateau counts once. Multiple peaks mean that improving mutations can lead to different endpoints.

In [11]:
print(f"Number of peaks: {landscape.n_lo}")

Number of peaks: 184


### 6.2 Roughness-to-slope ratio

For another view of ruggedness, we can calculate `r_s_ratio`. It compares deviations from an additive model with the average magnitude of its coefficients. Larger values indicate more residual roughness relative to the additive trend; a nearly zero fitted slope can give an infinite ratio.

In [12]:
roughness_to_slope = analysis.r_s_ratio(landscape)
print(f"Roughness-to-slope ratio: {roughness_to_slope:.4f}")

Roughness-to-slope ratio: 2.4772


### 6.3 Epistasis classification

A mutation’s effect can depend on the surrounding residues. This dependence, called *epistasis*, can create peaks and valleys when mutation effects change direction across backgrounds.

We can explore these patterns with `classify_epistasis()`, which classifies four-variant graph motifs:

| Category | Interpretation |
|---|---|
| `magnitude` | Mutation effects retain their direction across backgrounds. |
| `sign` | One mutation’s effect changes direction across backgrounds. |
| `reciprocal_sign` | Both mutations’ effects change direction across backgrounds. |

The output gives their proportions among the detected motifs. The `positive` and `negative` fields form a separate partition. GraphFLA automatically samples motifs for this large dataset; we set a seed to make the result repeatable.

In [13]:
epistasis = analysis.classify_epistasis(landscape, seed=42)
epistasis

EpistasisClassification(magnitude=0.4248912924324917, sign=0.3504951218620967, reciprocal_sign=0.22461358570541162, positive=0.9009051497397015, negative=0.09909485026029852)

### 6.4 Fitness-distance correlation

Finally, we can use `fdc` to check whether higher-fitness variants tend to lie closer to the best observed variant—a simple view of navigability. For protein sequences, distance is the number of differing amino-acid positions.

We use Spearman correlation to compare ranks. A negative value means higher fitness tends to occur closer to the optimum; a value near zero indicates little monotonic association. This trend does not guarantee an improving path from every variant.

In [14]:
fitness_distance_r = analysis.fdc(landscape, method="spearman")
print(f"Fitness-distance correlation: {fitness_distance_r:.4f}")

 - Calculating distances to global optimum using hamming_distance...


   - Distances to GO calculated and added as node attribute 'dist_go'.


Fitness-distance correlation: -0.1231


## 7. Running several analyses together

Once we’ve tried the metrics individually, we can collect them with `analysis.profile()`. Here, we combine r/s and fitness-distance correlation in a pandas Series. Setting `on_error="raise"` displays errors immediately.

In [15]:
analysis.profile(
    landscape,
    include=["r_s_ratio", "fdc"],
    on_error="raise",
    progress=False,
)

r_s_ratio    2.477187
fdc         -0.123074
dtype: float64

## 8. Analysis reference

For further exploration, we can choose from the **30 analysis functions** below. Each takes a built `landscape` as its first argument.

Walsh–Hadamard and higher-order regression are included for reference. High-order fits on the full GB1 dataset are computationally intensive, so we leave them out of the worked examples.

| Group | Function | Result |
| --- | --- | --- |
| Fitness | `fitness_distribution` | Dictionary of distribution summaries, including skewness, kurtosis, CV and Cauchy location. |
| Fitness | `fitness_effect_distribution` | List of signed effects for one mutation across matched backgrounds. |
| Ruggedness | `local_optima_ratio` | Local-peak count divided by variant count. |
| Ruggedness | `r_s_ratio` | Additive-model residual roughness divided by mean absolute coefficient. |
| Ruggedness | `autocorrelation` | Fitness autocorrelation along sampled walks. |
| Ruggedness | `gradient_intensity` | Mean absolute change on improving edges divided by mean fitness. |
| Correlation | `fdc` | Fitness-distance correlation: Spearman or Pearson. |
| Correlation | `neighbor_fitness_correlation` | Correlation between fitness and mean neighbour fitness: Pearson, Spearman or Kendall. |
| Correlation | `basin_fitness_correlation` | Correlation between peak fitness and greedy basin size: Spearman or Pearson. |
| Correlation | `fitness_flattening_index` | Trend in successive fitness gains along greedy walks: Spearman or Pearson. |
| Navigability | `global_optima_accessibility` | Fraction of variants with an improving route to the global optimum. |
| Navigability | `local_optima_accessibility` | DataFrame of accessibility values for specified local optima. |
| Navigability | `mean_distance_to_global_optimum` | Mean sequence/configuration distance to the global optimum. |
| Navigability | `mean_distance_to_local_optima` | DataFrame of mean sequence/configuration distances to specified local optima. |
| Navigability | `mean_path_length_to_global_optimum` | Mean shortest graph-path length, excluding unreachable starts. |
| Navigability | `mean_path_length_to_local_optima` | DataFrame of shortest-path summaries, excluding unreachable starts. `lo=None` selects the global optimum. |
| Robustness | `neutrality` | Fraction of neighbour pairs within the fitness-difference threshold. |
| Robustness | `evolvability_enhancing_mutations` | Fraction of improving edges that also increase mean neighbour fitness by more than epsilon. |
| Robustness | `single_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at one position; positive or negative tests. |
| Robustness | `all_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at all positions. |
| Epistasis | `idiosyncratic_index` | Normalized spread of one mutation’s effects across backgrounds. |
| Epistasis | `global_idiosyncratic_index` | Mean idiosyncratic index across eligible mutations. |
| Epistasis | `gamma` | Correlation of mutation effects across matched two-site backgrounds. |
| Epistasis | `gamma_star` | Consistency of mutation-effect signs across backgrounds. |
| Epistasis | `diminishing_returns_index` | Association of background fitness with mean beneficial gain: Pearson, Spearman or regression slope. |
| Epistasis | `increasing_costs_index` | Association of background fitness with mean deleterious cost: Pearson, Spearman or regression slope. |
| Epistasis | `higher_order_epistasis` | In-sample R² of a model including interactions up to the specified order. |
| Epistasis | `walsh_hadamard` | DataFrame of additive/interaction coefficients; the first retained protein sequence is the reference. |
| Epistasis | `classify_epistasis` | Object with magnitude, sign, reciprocal-sign, positive and negative epistasis proportions. |
| Epistasis | `extradimensional_bypass` | Object with bypass proportion, mean length and motif counts. |

### Data source

Wu NC, Dai L, Olson CA, Lloyd-Smith JO and Sun R (2016). [*Adaptation in protein fitness landscapes is facilitated by indirect paths*. eLife 5:e16965](https://doi.org/10.7554/eLife.16965).

We use the measured-variant CSV. The paper’s complete landscape also includes imputed variants, available in the [supplementary files](https://elifesciences.org/articles/16965/figures).